# Demo data and a live file

The command picture serves both. **Demo data** is a prepared story so a room can see every module before a camera is connected. **Live pipeline** shows a reading only after a file is read.

A person still confirms police, an ambulance, a signal plan, and a plate flag. The order stays on the screen.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from cityvigilance.sample import MODELS, run_model
from cityvigilance.analyze import build_event
print("sample models:", ", ".join(MODELS))

sample models: crowd, movement, weapon, accident, traffic, plate


## The prepared story

These are the numbers the desk plays in Demo data. Movement is always sudden so the review is visible. The plate is `DL01CV0001` and the complaint says it is not a real registration. The weapon line is a possible blade because every sample count is at least 40.

In [2]:
for name in MODELS:
    event = run_model(name, 0)
    print(f"{name:10} {event['cameraId']:12} {event['strength']:6} {event['finding']}")

crowd      cam-crowd    high   {'type': 'crowd', 'count': 46}
movement   cam-crowd    review {'type': 'movement', 'level': 'sudden', 'people': 46}
weapon     cam-crowd    review {'type': 'weapon', 'kind': 'blade'}
accident   cam-road     review {'type': 'accident', 'crash': True}
traffic    cam-junction high   {'type': 'traffic', 'arms': [{'name': 'North', 'queueMetres': 62, 'currentGreenSeconds': 20, 'greenSeconds': 31}, {'name': 'East', 'queueMetres': 22, 'currentGreenSeconds': 20, 'greenSeconds': 11}, {'name': 'South', 'queueMetres': 34, 'currentGreenSeconds': 25, 'greenSeconds': 17}, {'name': 'West', 'queueMetres': 14, 'currentGreenSeconds': 15, 'greenSeconds': 10}]}
plate      cam-junction review {'type': 'plate', 'text': 'DL01CV0001', 'listed': True, 'complaint': 'Demo complaint on file. This is not a real registration.'}


## The same modules on a file

A file feed counts people and vehicles. It does not invent a blade, a crash, or a plate. Movement is sudden only when the count jumps.

In [3]:
frames = [{"person": 4, "car": 1}, {"person": 11, "car": 6, "motorcycle": 2}]
for name in ("crowd", "movement", "weapon", "accident", "traffic", "plate"):
    event = build_event(name, frames, "2026-11-03T06:30:00Z")
    print(f"{name:10} {event['source']:6} {event['finding']}")

crowd      local  {'type': 'crowd', 'count': 11}
movement   local  {'type': 'movement', 'level': 'sudden', 'people': 11}
weapon     local  {'type': 'weapon', 'kind': 'none'}
accident   local  {'type': 'accident', 'crash': None}
traffic    local  {'type': 'traffic', 'basis': 'vehicles', 'arms': [{'name': 'In view', 'queueMetres': 0, 'vehicles': 8, 'currentGreenSeconds': 20, 'greenSeconds': 16}]}
plate      local  {'type': 'plate', 'text': '', 'listed': False, 'complaint': ''}


## Why V-JEPA is not the categoriser

V-JEPA and V-JEPA 2 learn a video embedding. The public action head is trained on Something-Something v2, which is hand and object actions such as "putting something into something". Those classes are not a crash, a weapon, congestion, or a number plate.

The model is hundreds of millions to about a billion parameters and is built for a GPU. This machine has no NVIDIA GPU, so the weights are not downloaded and no caption is invented.

V-JEPA can be a later video backbone after a head is trained on city events. A vision-language model can later write a sentence for the officer. That sentence still must not issue the order, and an API is allowed only if city video may leave the building.

Until then the desk shows the detector count, the movement review, and an explicit undecided line for weapon, crash, and plate.